# 🤖 Agentic RLVR: Multi-Turn ReAct, Sub-Agent Delegation & Tool Chaining with GRPO
## End-to-End Reinforcement Learning for `deepagents` & Sequential Tool-Calling — Kaggle T4 GPU

Real-world frontier agents (like **`deepagents`**, LangGraph, and coding harnesses) do not just make a single isolated tool call. They execute **sequential multi-turn trajectories**:
1. **Primary Orchestration**: Decompose complex goals into subtasks.
2. **Sub-Agent Delegation**: Delegate specialized domain tasks to sub-agents (`sql_analyst`, `python_engineer`).
3. **Virtual Filesystem & Tools**: Read/write virtual files, execute code, and query databases across multiple turns.
4. **Observation Feedback Loop**: Read intermediate outputs/tracebacks and self-correct.

This notebook implements **Multi-Turn Sequential ReAct & Sub-Agent RLVR** with Unsloth and GRPO on **Qwen2.5-1.5B-Instruct**.

---
### Environment Requirements:
- **Accelerator**: GPU T4 x2 (or GPU T4) — Select in Kaggle Settings
- **VRAM**: ~1.2 GB base model (fits comfortably within 16GB VRAM)
- **Internet**: Enabled


---
## 🔧 Section 0: Environment Setup & Hardware Verification


In [ ]:
# Install required libraries
!pip install -q --upgrade unsloth trl datasets transformers accelerate jsonschema


In [ ]:
# Pin to single GPU (cuda:0) to prevent multi-device sharding conflicts on Kaggle T4 x2
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")

if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    cap = torch.cuda.get_device_capability(0)
    props = torch.cuda.get_device_properties(0)
    vram = getattr(props, "total_memory", getattr(props, "total_mem", 0)) / 1e9
    print(f"Active GPU:      {device_name} (Compute Capability {cap[0]}.{cap[1]})")
    print(f"VRAM:            {vram:.1f} GB")
    
    if cap[0] < 7:
        print("\n" + "="*70)
        print(f"❌ INCOMPATIBLE GPU DETECTED: {device_name} (Compute Capability {cap[0]}.{cap[1]})")
        print("Unsloth requires Compute Capability >= 7.0 (e.g. T4, A100, L4).")
        print("Tesla P100 (Pascal, 6.0) is not supported.")
        print("\n👉 ACTION REQUIRED ON KAGGLE: Change Accelerator to 'GPU T4 x2' or 'GPU T4'.")
        print("="*70 + "\n")
        raise RuntimeError("Incompatible GPU: Please switch Kaggle Accelerator to GPU T4 in Settings.")
    else:
        print("✅ Single GPU pinned & architecture is compatible (Compute Capability >= 7.0)!")
else:
    print("⚠️  No GPU detected! Enable GPU in Settings → Accelerator → GPU T4 x2")


---
## 🛠️ Section 1: `deepagents` Subagent Registry & Virtual Filesystem Sandbox

We define a **hierarchical agent environment** with:
1. **Virtual Filesystem**: `write_file(path, content)`, `read_file(path)` to manage context.
2. **Subagent Delegations (`deepagents` style)**:
   - `delegate_subagent(subagent="sql_analyst", subtask="...")`
   - `delegate_subagent(subagent="python_engineer", subtask="...")`
3. **Direct Tools**: `calculator(expression)` for instant evaluations.


In [ ]:
import json
import sqlite3
import re
import sys
import io
import math
import contextlib

# ═══════════════════════════════════════════════════════════════════
# 1. Tool & Subagent Schemas (OpenAPI / JSONSchema specifications)
# ═══════════════════════════════════════════════════════════════════
TOOL_SCHEMAS = {
    "delegate_subagent": {
        "type": "object",
        "properties": {
            "subagent": {"type": "string", "enum": ["sql_analyst", "python_engineer"], "description": "Specialized subagent to delegate task to"},
            "subtask": {"type": "string", "description": "Detailed task description for the subagent"}
        },
        "required": ["subagent", "subtask"]
    },
    "write_file": {
        "type": "object",
        "properties": {
            "path": {"type": "string", "description": "File path in virtual filesystem"},
            "content": {"type": "string", "description": "Content to write to file"}
        },
        "required": ["path", "content"]
    },
    "read_file": {
        "type": "object",
        "properties": {
            "path": {"type": "string", "description": "File path to read from"}
        },
        "required": ["path"]
    },
    "calculator": {
        "type": "object",
        "properties": {
            "expression": {"type": "string", "description": "Mathematical expression"}
        },
        "required": ["expression"]
    }
}

# ═══════════════════════════════════════════════════════════════════
# 2. Virtual Sandbox with In-Memory DB & Virtual Filesystem
# ═══════════════════════════════════════════════════════════════════
class AgentSandbox:
    """Isolated execution environment for deepagents multi-turn workflows.
    
    Features:
    - In-memory SQLite database ('employees' table).
    - Restricted Python REPL with safe builtins.
    - In-memory virtual filesystem for inter-turn file read/write operations.
    """
    def __init__(self):
        self.files = {}
        self.conn = sqlite3.connect(":memory:")
        self._init_db()
        
    def _init_db(self):
        cursor = self.conn.cursor()
        cursor.execute("""
        CREATE TABLE employees (
            id INTEGER PRIMARY KEY,
            name TEXT,
            department TEXT,
            salary INTEGER,
            years_experience INTEGER
        );
        """)
        employees = [
            (1, 'Alice', 'Engineering', 120000, 6),
            (2, 'Bob', 'Engineering', 95000, 3),
            (3, 'Charlie', 'Marketing', 75000, 4),
            (4, 'Diana', 'Marketing', 110000, 8),
            (5, 'Evan', 'Sales', 65000, 2),
            (6, 'Fiona', 'Engineering', 140000, 10),
            (7, 'George', 'Sales', 85000, 5),
        ]
        cursor.executemany("INSERT INTO employees VALUES (?, ?, ?, ?, ?)", employees)
        self.conn.commit()

    def execute_sql(self, query: str) -> str:
        """Execute a SQL query inside the SQLite sandbox and return JSON rows.
        
        Examples:
            >>> sb = AgentSandbox()
            >>> sb.execute_sql("SELECT COUNT(*) FROM employees")
            '[[7]]'
        """
        try:
            cursor = self.conn.cursor()
            cursor.execute(query)
            rows = cursor.fetchall()
            return json.dumps(rows)
        except Exception as e:
            return f"SQL Error: {e}"

    def execute_python(self, code: str) -> str:
        """Execute Python code in a restricted namespace and return stdout / result.
        
        Examples:
            >>> sb = AgentSandbox()
            >>> sb.execute_python("result = sum([10, 20, 30])")
            '60'
        """
        buffer = io.StringIO()
        local_env = {"math": math, "files": self.files}
        safe_builtins = {
            "print": print, "range": range, "len": len, "sum": sum,
            "min": min, "max": max, "sorted": sorted, "int": int,
            "float": float, "str": str, "list": list, "dict": dict,
            "abs": abs, "round": round, "all": all, "any": any
        }
        try:
            with contextlib.redirect_stdout(buffer):
                exec(code, {"__builtins__": safe_builtins, "math": math}, local_env)
            output = buffer.getvalue().strip()
            if not output and "result" in local_env:
                output = str(local_env["result"])
            return output if output else "Executed successfully."
        except Exception as e:
            return f"Python Error: {e}"

    def execute(self, tool_name: str, args: dict) -> str:
        """Dispatch a tool or subagent call to the appropriate sandbox handler.
        
        Examples:
            >>> sb = AgentSandbox()
            >>> sb.execute("write_file", {"path": "test.txt", "content": "hello"})
            "Wrote 5 chars to virtual file 'test.txt'."
            >>> sb.execute("read_file", {"path": "test.txt"})
            'hello'
        """
        if tool_name == "delegate_subagent":
            sub = args.get("subagent")
            task = args.get("subtask", "")
            if sub == "sql_analyst":
                query_match = re.search(r"SELECT.*", task, re.IGNORECASE)
                query = query_match.group(0) if query_match else task
                return f"[sql_analyst]: {self.execute_sql(query)}"
            elif sub == "python_engineer":
                return f"[python_engineer]: {self.execute_python(task)}"
            return f"Error: Unknown subagent '{sub}'"
            
        elif tool_name == "write_file":
            path = args.get("path", "temp.txt")
            content = args.get("content", "")
            self.files[path] = content
            return f"Wrote {len(content)} chars to virtual file '{path}'."
            
        elif tool_name == "read_file":
            path = args.get("path", "")
            if path in self.files:
                return self.files[path]
            return f"Error: File '{path}' not found."
            
        elif tool_name == "calculator":
            expr = str(args.get("expression", "")).replace(",", "").replace("$", "")
            try:
                return str(eval(expr, {"__builtins__": None, "math": math}, {}))
            except Exception as e:
                return f"Error: {e}"
                
        return f"Error: Unknown tool '{tool_name}'"

print("deepagents Subagent Registry & Virtual Filesystem Sandbox initialized ✅")


---
## 📊 Section 2: Multi-Turn ReAct Dataset Construction

In multi-turn agent training, samples encompass both **initial goals** and **intermediate state transitions**:
```xml
<think>
Decompose problem into: 1) Query database via sql_analyst, 2) Process result.
</think>
<tool_call>
{"name": "delegate_subagent", "arguments": {"subagent": "sql_analyst", "subtask": "SELECT MAX(salary) FROM employees WHERE department = 'Engineering'"}}
</tool_call>
```


In [ ]:
from datasets import Dataset

DEEPAGENT_SYSTEM_PROMPT = """You are a frontier Orchestrator AI Agent built on deepagents. You solve complex multi-step workflows by delegating to specialized subagents and using virtual filesystem tools.

Available Subagents & Tools:
1. `delegate_subagent`: Delegate to specialized subagents. Args: {"subagent": "sql_analyst" | "python_engineer", "subtask": "string"}
2. `write_file`: Store intermediate outputs in virtual filesystem. Args: {"path": "string", "content": "string"}
3. `read_file`: Retrieve stored content from virtual filesystem. Args: {"path": "string"}
4. `calculator`: Fast arithmetic. Args: {"expression": "string"}

Format each turn with:
<think>
Your reasoning & multi-step plan.
</think>
<tool_call>
{"name": "tool_name", "arguments": {...}}
</tool_call>
Or when finished:
<answer>
final verified answer
</answer>"""

# Multi-step & subagent tasks
SEQUENTIAL_AGENT_DATA = [
    # Multi-step Subagent Delegations
    {
        "question": "Find the highest salary in Engineering and calculate what a 15% bonus on it would be.",
        "solution": "161000",
        "expected_subagent": "sql_analyst",
        "category": "multi_step"
    },
    {
        "question": "Get all employee salaries in Sales and compute their standard deviation.",
        "solution": "14142.14",
        "expected_subagent": "sql_analyst",
        "category": "multi_step"
    },
    {
        "question": "Query the total number of employees and write the count into 'stats.txt'.",
        "solution": "7",
        "expected_subagent": "sql_analyst",
        "category": "file_ops"
    },
    {
        "question": "What is the sum of the first 25 prime numbers? Delegate calculation to python_engineer.",
        "solution": "1060",
        "expected_subagent": "python_engineer",
        "category": "python_agent"
    },
    {
        "question": "Calculate 18 * 450 + (12000 / 24) - 85 using calculator.",
        "solution": "8515",
        "expected_subagent": "calculator",
        "category": "calculator"
    }
]

# Build training dataset
train_records = []
for _ in range(50): # 250 samples
    for item in SEQUENTIAL_AGENT_DATA:
        train_records.append({
            "prompt": [
                {"role": "system", "content": DEEPAGENT_SYSTEM_PROMPT},
                {"role": "user", "content": item["question"]}
            ],
            "solution": item["solution"],
            "expected_subagent": item["expected_subagent"]
        })

agent_dataset = Dataset.from_list(train_records)
print(f"Sequential multi-turn dataset created: {len(agent_dataset)} samples ✅")


---
## 🎯 Section 3: Sequential & Sub-Agent Trajectory Verifier Suite

We define verifiers for **sequential tool calling and subagent orchestration**:
1. 🎯 **`trajectory_execution_reward`**: Evaluates execution correctness in the live sandbox.
2. 🛡️ **`subagent_delegation_reward`**: Verifies correct subagent selection (`sql_analyst` vs `python_engineer`).
3. 📐 **`trajectory_format_reward`**: Validates `<think>`, `<tool_call>`, and `<answer>` XML tags.


In [ ]:
def extract_completion_text(completion) -> str:
    """Extract clean string from string, dict, or conversational message lists.
    
    Examples:
        >>> extract_completion_text([{'role': 'assistant', 'content': '<answer>10</answer>}])
        '<answer>10</answer>'
    """
    if isinstance(completion, str):
        return completion
    elif isinstance(completion, list) and len(completion) > 0:
        if isinstance(completion[-1], dict) and "content" in completion[-1]:
            return completion[-1]["content"]
        elif isinstance(completion[0], dict) and "content" in completion[0]:
            return completion[0]["content"]
        return str(completion[-1])
    elif isinstance(completion, dict) and "content" in completion:
        return completion["content"]
    return str(completion)

def parse_agent_turn(text: str):
    """Parse an agent response into thought, tool call dict, and final answer.
    
    Examples:
        >>> text = '<think>T</think><tool_call>{"name": "calc", "arguments": {}}</tool_call>'
        >>> think, tool, ans = parse_agent_turn(text)
        >>> think
        'T'
        >>> tool['name']
        'calc'
    """
    think = re.search(r"<think>(.*?)</think>", text, re.DOTALL)
    tool = re.search(r"<tool_call>(.*?)</tool_call>", text, re.DOTALL)
    ans = re.search(r"<answer>(.*?)</answer>", text, re.DOTALL)
    
    tool_dict = None
    if tool:
        try:
            tool_dict = json.loads(tool.group(1).strip())
        except Exception:
            tool_dict = "INVALID_JSON"
            
    return (
        think.group(1).strip() if think else None,
        tool_dict,
        ans.group(1).strip() if ans else None
    )

# 1. Trajectory Execution Verifier
def trajectory_execution_reward(prompts, completions, solution, **kwargs):
    """Score trajectory execution correctness against sandbox state or numerical match.
    
    Args:
        prompts: Input prompts from GRPOTrainer.
        completions: Model generated completions.
        solution: Target verified numerical or string answer.
        
    Returns:
        List[float]: 1.0 for exact match, 0.8 for successful tool execution output, 0.0 otherwise.
    """
    rewards = []
    for completion, sol in zip(completions, solution):
        text = extract_completion_text(completion)
        _, tool_dict, ans = parse_agent_turn(text)
        sol_clean = str(sol).replace(",", "").replace("$", "").strip().lower()
        
        score = 0.0
        if ans:
            ans_clean = str(ans).replace(",", "").replace("$", "").strip().lower()
            try:
                if abs(float(ans_clean) - float(sol_clean)) < 0.05:
                    score = 1.0
            except ValueError:
                if ans_clean == sol_clean:
                    score = 1.0
                    
        if score == 0.0 and isinstance(tool_dict, dict):
            sb = AgentSandbox()
            out = sb.execute(tool_dict.get("name"), tool_dict.get("arguments", {}))
            if sol_clean in str(out).lower():
                score = 0.8
                
        rewards.append(score)
    return rewards

# 2. Subagent Delegation Verifier
def subagent_delegation_reward(prompts, completions, expected_subagent, **kwargs):
    """Score whether the agent delegated the task to the expected specialized subagent.
    
    Args:
        prompts: Input prompts from GRPOTrainer.
        completions: Model generated completions.
        expected_subagent: Target subagent ('sql_analyst', 'python_engineer', 'calculator').
        
    Returns:
        List[float]: 1.0 for exact subagent/tool match, 0.5 for valid tool schema, 0.0 otherwise.
    """
    rewards = []
    for completion, exp_sub in zip(completions, expected_subagent):
        text = extract_completion_text(completion)
        _, tool_dict, _ = parse_agent_turn(text)
        
        if isinstance(tool_dict, dict):
            name = tool_dict.get("name")
            args = tool_dict.get("arguments", {})
            if name == "delegate_subagent" and args.get("subagent") == exp_sub:
                rewards.append(1.0)
            elif name == exp_sub:
                rewards.append(1.0)
            elif name in TOOL_SCHEMAS:
                rewards.append(0.5)
            else:
                rewards.append(0.0)
        else:
            rewards.append(0.0)
    return rewards

# 3. Trajectory Format Verifier
def trajectory_format_reward(prompts, completions, **kwargs):
    """Score XML formatting compliance (<think>, <tool_call>, <answer>).
    
    Returns:
        List[float]: 1.0 for valid think+tool/answer, 0.5 for tool/answer without think, 0.0 for none.
    """
    rewards = []
    for completion in completions:
        text = extract_completion_text(completion)
        has_think = bool(re.search(r"<think>.*?</think>", text, re.DOTALL))
        has_tool = bool(re.search(r"<tool_call>.*?</tool_call>", text, re.DOTALL))
        has_ans = bool(re.search(r"<answer>.*?</answer>", text, re.DOTALL))
        
        if has_think and (has_tool or has_ans):
            rewards.append(1.0)
        elif has_tool or has_ans:
            rewards.append(0.5)
        else:
            rewards.append(0.0)
    return rewards

print("Sequential & Subagent Verifier Suite defined with rich docstrings ✅")


---
## 🧠 Section 4: Model & LoRA Adapter Setup


In [ ]:
from unsloth import FastLanguageModel, is_bfloat16_supported

MODEL_NAME = "unsloth/Qwen2.5-1.5B-Instruct"
MAX_SEQ_LENGTH = 1024

print(f"📦 Loading {MODEL_NAME}...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    device_map="cuda:0",
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    use_gradient_checkpointing="unsloth",
)

print(f"✅ Model loaded with LoRA on cuda:0")


---
## 🚀 Section 5: GRPO Training for Subagent & Multi-Turn Agents


In [ ]:
from trl import GRPOConfig, GRPOTrainer

training_args = GRPOConfig(
    output_dir="./grpo-deepagent-qwen",
    num_train_epochs=1,
    max_steps=200,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_generations=4,
    max_completion_length=384,
    learning_rate=5e-6,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    beta=0.04,
    logging_steps=5,
    save_steps=50,
    save_total_limit=2,
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    gradient_checkpointing=True,
    seed=42,
    report_to="none",
)

reward_funcs = [
    trajectory_execution_reward,
    subagent_delegation_reward,
    trajectory_format_reward,
]

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    args=training_args,
    train_dataset=agent_dataset,
    reward_funcs=reward_funcs,
)

print("GRPOTrainer configured for deepagents multi-turn RLVR ✅")


In [ ]:
print("🚀 Starting Agentic GRPO Training...")
trainer.train()
print("✅ Training Complete!")


---
## 🎮 Section 6: Interactive Multi-Turn ReAct Trajectory Engine

Test sequential multi-turn tool calling and subagent delegation in the live interactive sandbox!


In [ ]:
def run_deepagent_interactive(goal_instruction: str, max_turns: int = 4):
    """Runs a full multi-turn sequential ReAct loop with sandbox execution.
    
    Args:
        goal_instruction: Complex user task requiring tool calling or subagents.
        max_turns: Maximum allowed interaction rounds.
        
    Examples:
        >>> run_deepagent_interactive('Find max salary in Engineering and calculate 15% bonus.')
    """
    FastLanguageModel.for_inference(model)
    sandbox = AgentSandbox()
    
    messages = [
        {"role": "system", "content": DEEPAGENT_SYSTEM_PROMPT},
        {"role": "user", "content": goal_instruction}
    ]
    
    print("="*75)
    print(f"🎯 USER GOAL: {goal_instruction}")
    print("="*75)
    
    for turn in range(max_turns):
        print(f"\n--- [Turn {turn+1}] ---")
        prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt_text, return_tensors="pt").to("cuda:0")
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=384,
                temperature=0.2,
                do_sample=True,
                use_cache=True
            )
            
        response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
        think, tool_dict, ans = parse_agent_turn(response)
        
        if think:
            print(f"💭 Thought: {think}")
            
        if ans:
            print(f"\n🏁 Final Answer: {ans}")
            break
            
        if isinstance(tool_dict, dict):
            tool_name = tool_dict.get("name")
            args = tool_dict.get("arguments", {})
            print(f"🛠️ Tool Action: [{tool_name}] -> {json.dumps(args)}")
            
            # Execute in live sandbox
            observation = sandbox.execute(tool_name, args)
            print(f"📥 Observation: {observation}")
            
            # Append to history
            messages.append({"role": "assistant", "content": response})
            messages.append({"role": "user", "content": f"<observation>\n{observation}\n</observation>"})
        else:
            print(f"⚠️ Output: {response}")
            break
            
    print("="*75)

# Run multi-turn demonstration
run_deepagent_interactive("Find the employee with highest salary in Engineering and calculate a 15% bonus.")
